<a href="https://colab.research.google.com/github/Prianka-Mukhopadhyay/voyage-analytics/blob/main/Voyage_Analytics_Copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**VOYAGE ANALYTICS**
### **Group Members**
####Prianka Mukhopadhyay
####Aayush Kaul
####Sri Vaishnavi Thalloju
####Om Mane


#**PROBLEM STATEMENTS**
####Build a regression model to predict the prices of a flight acccurately.
####Deploy a classification model to acccurately identify user's gender.
####Build a recommendation model to provide hotel suggestions based on user preferrences and historical data.
####Build a streamlit application to display insights using travel recommendation model.

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
!pip install streamlit

In [ ]:
# import necessary functions
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# import data
flights = pd.read_csv('flights.csv')
users = pd.read_csv('users.csv')
hotels = pd.read_csv('hotels.csv')

In [ ]:
# Null value traetment
flights.isnull().sum()

In [ ]:
users.isnull().sum()

In [ ]:
hotels.isnull().sum()

In [ ]:
# Merge dataset for recommendations
users_flights = users.merge(flights, left_on='code', right_on='userCode')

In [ ]:
# Separate x and y for price prediction.
x= flights.drop(['price'],axis=1)
y= flights['price']

In [ ]:
# Separate cat and con from x
x_cat= x.select_dtypes(object)
x_con= x.select_dtypes(np.number)

In [ ]:
# Check combined cat and con from dataframe.
x_cat.head()

In [ ]:
x_con.head()

In [ ]:
x = pd.concat([x_cat, x_con], axis=1)
x.head()

In [ ]:
# Preprocess data
# Feature Encoding
le = LabelEncoder()
x_cat = x_cat.apply(le.fit_transform)

In [ ]:
# Feature Scaling
ss= StandardScaler()
x_con = pd.DataFrame(ss.fit_transform(x_con), columns=x_con.columns)

In [ ]:
# Remove outliers and reset the index
out =[]
for i in x_con.columns:
  o1 = [(x_con[i]< -3) | (x_con[i]>3)]
  out.append(o1)
out

In [ ]:
# Extract the boolean Series from the nested list structure in 'out'
outlier_masks = [mask[0] for mask in out]

# Combine the boolean masks to find rows where at least one column is an outlier
# This creates a DataFrame of boolean masks and then checks if any column is True for each row
combined_outlier_mask = pd.concat(outlier_masks, axis=1).any(axis=1)

# Get the indices of the rows identified as outliers
outliers = x_con.index[combined_outlier_mask].tolist()

outliers # Display the list of outlier indices

In [ ]:
x.shape

In [ ]:
y.shape

In [ ]:
import statsmodels.api as sm

In [ ]:
#Feature Selection

# First, ensure 'x' is the correctly preprocessed data by combining encoded x_cat and scaled x_con
x_for_modeling = pd.concat([x_cat, x_con], axis=1)

# To resolve the NameError, we need an initial model to determine the first 'c' to drop.
# Perform an initial split and fit to define an 'initial_model'
xtrain_init, _, ytrain_init, _ = train_test_split(x_for_modeling, y, test_size=0.2, random_state=21)

# Add a constant for the OLS model (sm was imported in a previous cell)
X_train_const_init = sm.add_constant(xtrain_init)

# Fit an initial OLS model to get the p-values for the first drop
initial_model = sm.OLS(ytrain_init, X_train_const_init).fit()

# Now, 'initial_model' is defined and we can get the first 'c' to drop
c = initial_model.pvalues.sort_values().index[-1] # last column to drop based on initial model

print(f"Initial model R-squared adjusted: {round(initial_model.rsquared_adj, 3)}")
print(f"Feature with highest p-value (to drop first): '{c}'")

# Drop that column from the features used for the *next* split and model fitting
x = x_for_modeling.drop(labels=c, axis=1)

# Now, split the data again with the modified 'x'
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=21)

# Add constant and fit the OLS model with the reduced feature set
ols=sm.OLS(ytrain,sm.add_constant(xtrain))
model=ols.fit() # 'model' is now defined for the rest of the cell

score=round(model.rsquared_adj,3)

# This 'c' will be the next column to drop based on the newly fitted 'model'
c=model.pvalues.sort_values().index[-1]

print('Next column to drop (if continuing the process):', c)
print('R-squared adjusted after first feature drop:', score)

In [ ]:
# Visualisation
for i in x_con.columns:
  sns.histplot(data =x.head(10), x=str(i), kde =True)
  plt.show()

In [ ]:
plot_df = pd.concat([x, y], axis=1)
for i in x_con.columns:
  sns.scatterplot(data = plot_df.head(10), x=str(i), y='price')
  plt.show()

In [ ]:
plot_df = pd.concat([x, y], axis=1)
# Filter x_cat.columns to only include columns that are present in the current x DataFrame
categorical_cols_for_plotting = [col for col in x_cat.columns if col in x.columns]

for i in categorical_cols_for_plotting:
  sns.boxplot(data = plot_df.head(10), x=str(i), y='price')
  plt.show()

In [ ]:
#Apply XG Boost
from xgboost import XGBRegressor

In [ ]:
xgb = XGBRegressor()
model = xgb.fit(xtrain,ytrain)

In [ ]:
ypred_train = model.predict(xtrain)
ypred_test = model.predict(xtest)

In [ ]:
print("Training accuracy is :", model.score(xtrain,ytrain))
print("Testing accuracy is :", model.score(xtest,ytest))

In [ ]:
# Install MLFLOW
!pip install mlflow

In [ ]:
import mlflow
import mlflow.xgboost
mlflow.set_experiment("voyage-analytics-models")

In [ ]:
mlflow.autolog()  # Auto-logs everything
with mlflow.start_run(run_name="flight-price-xgb"):
    model = xgb.fit(xtrain, ytrain)
    ypred_test = model.predict(xtest)
    # Custom metrics if needed
    from sklearn.metrics import r2_score, mean_squared_error
    r2 = r2_score(ytest, ypred_test)
    mse = mean_squared_error(ytest, ypred_test)
    rmse = np.sqrt(mse) # Calculate RMSE manually
    mlflow.log_metric("r2_test", r2)
    mlflow.log_metric("rmse_test", rmse)
mlflow.end_run()

In [ ]:
# CLASSIFICATION MODEL

In [ ]:
# Separate x and y for gender classification.
x = users.drop(['gender'], axis=1)
y = users['gender']

In [ ]:
users['gender'].value_counts()

In [ ]:
# Separate cat and con from X
cat=[]
con=[]
for i in x.columns:
  if x[i].dtypes==object:
    cat.append(i)
  else:
    con.append(i)

In [ ]:
xcat=x[cat]
xcon=x[con]

In [ ]:
for i in xcat.columns:
  xcat[i]=le.fit_transform(xcat[i])

In [ ]:
xcon=pd.DataFrame(ss.fit_transform(xcon),columns=con)

In [ ]:
x=xcon.join(xcat)
x.head()

In [ ]:
# Label Encode Y since  it is a categorical output with related categoric outputs.
y=le.fit_transform(y)
y=pd.DataFrame(y,columns=['gender'])
y.head()

In [ ]:
# Remove outliers and reset the index.
out=[]
for i in xcon.columns:
  o1=xcon[(xcon[i]<-3)|(xcon[i]>3)].index
  out.extend(o1)
out=list(set(out))

In [ ]:
print(out)

In [ ]:
x.shape

In [ ]:
y.shape

In [ ]:
#  Reduce biasedness of data using SMOTE
from imblearn.over_sampling import SMOTE

In [ ]:
x_smote,y_smote=SMOTE().fit_resample(x,y)

In [ ]:
y_smote.value_counts()

In [ ]:
y_smote=pd.DataFrame(y_smote,columns=['gender'])
y

In [ ]:
# Perform train test split
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
#  Feature Selection
# Sequential Feature Selection
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Define the model RFC
clf = RandomForestClassifier(n_estimators=10, random_state=42) # Added random_state for reproducibility
# Initialize SFS with the classifier for feature selection for the classification task
sfs=SequentialFeatureSelector(clf,n_features_to_select='auto',direction='forward',cv=4)
sfs.fit(x_smote,y_smote.values.ravel()) # .values.ravel() to handle (n,1) vs (n,) shape mismatch

In [ ]:
print("Selected features are:", sfs.get_support(indices=True))

In [ ]:
# Error Evaluation
# Extract selected features for training and testing based on sfs.get_support()
selected_features_indices = sfs.get_support(indices=True)

# Prepare data with selected features
x_smote_selected = x_smote.iloc[:, selected_features_indices]
xtest_selected = xtest.iloc[:, selected_features_indices]

# Train the RandomForestClassifier (clf) with the SMOTE'd data and selected features
clf.fit(x_smote_selected, y_smote.values.ravel())

# Make predictions for the classification task
ypred_train_clf = clf.predict(x_smote_selected)
ypred_test_clf = clf.predict(xtest_selected)

In [ ]:
# Evaluate the classification model using the correct predictions and true labels
print("Training accuracy score is",accuracy_score(y_smote.values.ravel(), ypred_train_clf))
print("Testing accuracy score is",accuracy_score(ytest.values.ravel(), ypred_test_clf))
print('***')
print("Training confusion matrix is")
print(confusion_matrix(y_smote.values.ravel(), ypred_train_clf))
print("Testing confusion matrix is")
print(confusion_matrix(ytest.values.ravel(), ypred_test_clf))

In [ ]:
from sklearn.metrics import f1_score

In [ ]:
with mlflow.start_run(run_name="user-gender-rf"):
    # Using the SMOTE'd and selected features for training
    clf.fit(x_smote_selected, y_smote.values.ravel())
    mlflow.sklearn.log_model(clf, "gender_model")
    # Logging f1_score for the test set
    mlflow.log_metric("f1_test", f1_score(ytest.values.ravel(), clf.predict(xtest_selected), average='weighted'))

In [ ]:
# Recommendation Model

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Feature Engineering
user_hotel_matrix = hotels.pivot_table(
    index='userCode',
    columns='name',
    values='total',
    aggfunc='sum'
).fillna(0)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
user_similarity = cosine_similarity(user_hotel_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_hotel_matrix.index,
    columns=user_hotel_matrix.index
)

In [ ]:
def recommend_hotels(user_id):

    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]

    similar_users_data = user_hotel_matrix.loc[similar_users.index]

    recommended_hotels = similar_users_data.sum().sort_values(ascending=False)

    already_visited = user_hotel_matrix.loc[user_id]
    recommended_hotels = recommended_hotels[already_visited == 0]

    return recommended_hotels.head(5)

In [ ]:
def recommend_by_place(place):
    filtered = df[df['place'] == place]
    return filtered.sort_values(by='total', ascending=False)[['name', 'place', 'total']].drop_duplicates().head(5)

In [ ]:
def hybrid_recommendation(user_id):

    # Step 1: collaborative
    collab = recommend_hotels(user_id).index.tolist()

    # Step 2: user's preferred place
    user_places = df[df['userCode'] == user_id]['place'].mode()[0]

    content = df[df['place'] == user_places]['name'].unique().tolist()

    # combine
    final = list(set(collab + content))

    return final[:5]

In [ ]:
mlflow.set_experiment("Travel_Recommendation_Model")

In [ ]:
import mlflow.pyfunc

class HybridRecommenderModel(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        # Load the clean hotels data from artifact
        self.hotels_df = pd.read_csv(context.artifacts["data"])

        # Re-create user_hotel_matrix
        self.user_hotel_matrix = self.hotels_df.pivot_table(
            index='userCode',
            columns='name',
            values='total',
            aggfunc='sum'
        ).fillna(0)

        # Re-create user_similarity_df
        # Check if user_hotel_matrix is not empty before calculating similarity
        if not self.user_hotel_matrix.empty:
            self.user_similarity = cosine_similarity(self.user_hotel_matrix)
            self.user_similarity_df = pd.DataFrame(
                self.user_similarity,
                index=self.user_hotel_matrix.index,
                columns=self.user_hotel_matrix.index
            )
        else:
            self.user_similarity_df = pd.DataFrame() # Empty DataFrame if no data

    def _recommend_hotels(self, user_id):
        # Handle cases where user_id might not be in the matrix or similarity df is empty
        if self.user_similarity_df.empty or user_id not in self.user_similarity_df.columns:
            return pd.Series(dtype=object) # Return empty series if user not found or no data

        similar_users = self.user_similarity_df[user_id].sort_values(ascending=False)[1:6]

        # Filter out users that might not be in the matrix (though unlikely if user_id is from original data)
        similar_users_in_matrix = similar_users.index.intersection(self.user_hotel_matrix.index)
        if similar_users_in_matrix.empty:
            return pd.Series(dtype=object)

        similar_users_data = self.user_hotel_matrix.loc[similar_users_in_matrix]

        recommended_hotels = similar_users_data.sum().sort_values(ascending=False)

        # Ensure already_visited is based on the correct user_id in the matrix
        if user_id in self.user_hotel_matrix.index:
            already_visited = self.user_hotel_matrix.loc[user_id]
            # Only recommend hotels not already visited by the user
            recommended_hotels = recommended_hotels[already_visited == 0]
        else:
            # If user has no history, recommend top hotels from similar users
            pass # recommended_hotels already contains these

        return recommended_hotels.head(5)

    def _recommend_by_place(self, place):
        filtered = self.hotels_df[self.hotels_df['place'] == place]
        return filtered.sort_values(by='total', ascending=False)[['name', 'place', 'total']].drop_duplicates().head(5)

    def predict(self, context, model_input):
        user_id = model_input['userCode'].iloc[0] # Assuming model_input is a DataFrame with 'userCode'

        # Step 1: collaborative filtering recommendations
        collab_recs = self._recommend_hotels(user_id)
        collab_hotel_names = collab_recs.index.tolist()

        # Step 2: content-based recommendations (user's preferred place)
        user_history = self.hotels_df[self.hotels_df['userCode'] == user_id]
        if not user_history.empty:
            user_preferred_place = user_history['place'].mode()
            if not user_preferred_place.empty:
                user_preferred_place = user_preferred_place[0]
                content_recs = self.hotels_df[self.hotels_df['place'] == user_preferred_place]['name'].unique().tolist()
            else:
                content_recs = []
        else:
            content_recs = [] # No history for content-based

        # Combine and ensure uniqueness, then return top 5
        final_recommendations = list(set(collab_hotel_names + content_recs))

        return final_recommendations[:5]

In [ ]:
with mlflow.start_run() as run:
    global recommender_model_run_id
    recommender_model_run_id = run.info.run_id

    # Save dataset as artifact
    hotels.to_csv("hotels_clean.csv", index=False)

    # Log parameters
    mlflow.log_param("model_type", "Hybrid Collaborative + Content")
    mlflow.log_param("similarity_metric", "cosine")
    mlflow.log_param("input_data", "userCode + hotel history")

    # Log basic metrics
    mlflow.log_metric("num_users", hotels['userCode'].nunique())
    mlflow.log_metric("num_hotels", hotels['name'].nunique())
    mlflow.log_metric("avg_bookings_per_user", hotels.groupby('userCode').size().mean())

    # Log model
    mlflow.pyfunc.log_model(
        artifact_path="recommender_model",
        python_model=HybridRecommenderModel(),
        artifacts={"data": "hotels_clean.csv"}
    )

In [ ]:
# Using the programmatically retrieved run ID
model = mlflow.pyfunc.load_model(f"runs:/{recommender_model_run_id}/recommender_model")

In [ ]:
import pandas as pd
test_input = pd.DataFrame({'userCode': [101]})
model.predict(test_input)

In [ ]:
# Model creation for all the 3.
import joblib

In [ ]:
# Regression
joblib.dump(model, 'model_price.pkl')
print("model_price.pkl saved")

In [ ]:
# Classification
joblib.dump(clf, 'model_gender.pkl')
print("model_gender.pkl saved")

In [ ]:
# Recommendation
joblib.dump(model, 'model_hybrid.pkl')
print("model_hybrid.pkl saved")

In [ ]:
import os
print(os.listdir())